# Extra experiments

In [1]:
"""
Author: TMJ
Date: 2025-03-12 19:33:20
LastEditors: TMJ
LastEditTime: 2025-03-23 23:14:08
Description: 请填写简介
"""

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from rdkit.Chem import Draw
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold
import matplotlib.patches as patches

from dative_chemprop.models.DativeCGR import DativeCGR
from dative_chemprop.predictor import FengMichaelPredictor
from dative_chemprop.utils import *
from skfp.distances import dice_count_similarity, tanimoto_count_similarity
from sklearn.metrics.pairwise import PAIRWISE_DISTANCE_FUNCTIONS
from skfp.fingerprints import ECFPFingerprint


from seaborn import JointGrid
import matplotlib.patches as mpatches

fpgen = ECFPFingerprint(radius=3, fp_size=2048, count=True)
plt.rcParams["font.family"] = "Arial"
torch.set_float32_matmul_precision("high")
chemprop_dir = Path.cwd()
input_path = chemprop_dir / "data" / "ML_data_with_pesudo_state.csv"
dataset = pd.read_csv(input_path, index_col=0)

rxn_smiles_columns = [
    "rxn_smiles",
    "rxn_active_smiles",
    "dative_rxn_smiles",
    "dative_active_rxn_smiles",
]
target_columns = ["ddG (kcal/mol)"]

published_path = Path.cwd() / "data" / "published_data.csv"
published_dataset = pd.read_csv(published_path, index_col=0)
published_dataset["donor_class_code"] = published_dataset["donor_smiles"].apply(
    lambda x: get_reactant_class(x, donor_templates_mapping)
)
published_dataset["donor_indice"] = published_dataset["donor_smiles"].apply(
    lambda x: get_reactant_indice(x, donor_templates_mapping)
)
published_dataset["acceptor_class_code"] = published_dataset["acceptor_smiles"].apply(
    lambda x: get_reactant_class(x, acceptor_templates_mapping)
)
published_dataset["acceptor_indice"] = published_dataset["acceptor_smiles"].apply(
    lambda x: get_reactant_indice(x, acceptor_templates_mapping)
)
tempd = published_dataset.copy()
background_data = tempd.loc[tempd["ee (%)"] > 80].copy()
background_data.loc[:, "Ligand Type"] = background_data["ligand_type"].to_numpy()
background_data.loc[:, "Metal Type"] = background_data["is_rare_earth"].to_numpy()
d = background_data.copy()
background_data.rename(
    columns={
        "Metal": "Metal Salt",
        "Metal Type": "Metal Salt Type",
    },
    inplace=True,
)

random_seed = 42
dataset

/home/tmj/miniforge3/envs/feng/lib/python3.11/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


,rxn_smiles,ddG (kcal/mol),ee (%),acceptor_smiles,donor_smiles,donor_active_smiles,reactants_couples_smiles,reactants_active_couples_smiles,additives_smiles,solvents_smiles,...,pure_active_rxn_smiles,rxn_active_smiles,Ref.(DOI),temperature (C),pesudo_active_donor_rxn_smiles,donor_smiles_pesudo_active,no_extra_dative_rxn_smiles,no_extra_dative_pesudo_active_donor_rxn_smiles,yield (%),de (%)
0,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,2.279682,95,Cc1nn(c(c1)C)C(=O)/C=C/c1ccc(cc1)C(F)(F)F,CCOC(=O)CC(=O)NCc1ccccc1,CCOC(=O)C=C([O-])NCC1=CC=CC=C1,Cc1nn(c(c1)C)C(=O)/C=C/c1ccc(cc1)C(F)(F)F.CCOC...,Cc1nn(c(c1)C)C(=O)/C=C/c1ccc(cc1)C(F)(F)F.CCOC...,CCN(CC)CC,ClCCl,...,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,10.1002/chem.201603056,40,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,[CH2-]COC(=O)CC(=O)NCC1=CC=CC=C1,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,96.0,90.000000
1,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,2.063894,93,Cc1nn(c(c1)C)C(=O)/C=C/c1ccc(c(c1)Cl)Cl,CCOC(=O)CC(=O)NCc1ccccc1,CCOC(=O)C=C([O-])NCC1=CC=CC=C1,Cc1nn(c(c1)C)C(=O)/C=C/c1ccc(c(c1)Cl)Cl.CCOC(=...,Cc1nn(c(c1)C)C(=O)/C=C/c1ccc(c(c1)Cl)Cl.CCOC(=...,CCN(CC)CC,ClCCl,...,Cc1nn([C:7](=[O:8])/[CH:19]=[CH:20]/[c:21]2[cH...,Cc1nn([C:7](=[O:8])/[CH:19]=[CH:20]/[c:21]2[cH...,10.1002/chem.201603056,40,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,[CH2-]COC(=O)CC(=O)NCC1=CC=CC=C1,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,97.0,84.000000
2,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,1.832202,90,Cc1nn(c(c1)C)C(=O)/C=C/c1ccc2c(c1)cccc2,CCOC(=O)CC(=O)NCc1ccccc1,CCOC(=O)C=C([O-])NCC1=CC=CC=C1,Cc1nn(c(c1)C)C(=O)/C=C/c1ccc2c(c1)cccc2.CCOC(=...,Cc1nn(c(c1)C)C(=O)/C=C/c1ccc2c(c1)cccc2.CCOC(=...,CCN(CC)CC,ClCCl,...,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,10.1002/chem.201603056,40,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,[CH2-]COC(=O)CC(=O)NCC1=CC=CC=C1,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,94.0,81.981982
3,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,2.163032,94,Cc1nn(c(c1)C)C(=O)/C=C/c1ccccc1Br,CCOC(=O)CC(=O)NCc1ccccc1,CCOC(=O)C=C([O-])NCC1=CC=CC=C1,Cc1nn(c(c1)C)C(=O)/C=C/c1ccccc1Br.CCOC(=O)CC(=...,Cc1nn(c(c1)C)C(=O)/C=C/c1ccccc1Br.CCOC(=O)C=C(...,CCN(CC)CC,ClCCl,...,Cc1nn([C:7](=[O:8])/[CH:19]=[CH:20]/[c:21]2[cH...,Cc1nn([C:7](=[O:8])/[CH:19]=[CH:20]/[c:21]2[cH...,10.1002/chem.201603056,40,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,[CH2-]COC(=O)CC(=O)NCC1=CC=CC=C1,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,87.0,70.149254
4,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,2.421718,96,Cc1nn(c(c1)C)C(=O)/C=C/c1ccccc1,CCOC(=O)CC(=O)NCc1ccccc1,CCOC(=O)C=C([O-])NCC1=CC=CC=C1,Cc1nn(c(c1)C)C(=O)/C=C/c1ccccc1.CCOC(=O)CC(=O)...,Cc1nn(c(c1)C)C(=O)/C=C/c1ccccc1.CCOC(=O)C=C([O...,CCN(CC)CC,ClCCl,...,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,10.1002/chem.201603056,40,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,[CH2-]COC(=O)CC(=O)NCC1=CC=CC=C1,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,Cc1nn([C:17](=[O:18])/[CH:19]=[CH:20]/[c:21]2[...,97.0,81.981982
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1646,[O:8]=[C:7]1/[C:6](=[CH:5]/[c:23]2[cH:24][cH:2...,2.243283,95,O=C1/C(=C/c2ccc3c(c2)cccc3)/c2c(N1Cc1ccccc1)cccc2,SCC=O,O=CC[S-],O=C1/C(=C/c2ccc3c(c2)cccc3)/c2c(N1Cc1ccccc1)cc...,O=C1/C(=C/c2ccc3c(c2)cccc3)/c2c(N1Cc1ccccc1)cc...,NaN,ClCCl,...,[O:8]=[C:7]1/[C:6](=[CH:5]/[c:23]2[cH:24][cH:2...,[O:8]=[C:7]1/[C:6](=[CH:5]/[c:23]2[cH:24][cH:2...,10.1002/adsc.201400964,35,[O:8]=[C:7]1/[C:6](=[CH:5]/[c:23]2[cH:24][cH:2...,O=C[CH-]S,[O:8]=[C:7]1/[C:6](=[CH:5]/[c:23]2[cH:24][cH:2...,[O:8]=[C:7]1/[C:6](=[CH:5]/[c

In [2]:
batch_size = 128
train_smis = dataset[rxn_smiles_columns].values
train_y = dataset[target_columns].values
base_model = DativeCGR(
    model_name=f"augmented_base_model",
    seed=random_seed,
)
if not os.path.exists(base_model.model_path):
    base_model.fit(
        train_smis.flatten(),
        train_y.repeat(len(rxn_smiles_columns), axis=0),
        batch_size=batch_size,
        num_workers=16,
        max_epochs=600,
    )


def train_ood_model(
    base_model: DativeCGR,
    new_model_tag: str,
    target_acceptor: str,
    target_donor: str,
    train_smis: np.ndarray,
    train_y: np.ndarray,
    batch_size: int = 128,
    num_workers: int = 16,
    max_epochs: int = 200,
):
    new_model_name = f"{base_model.model_name}_{new_model_tag}"
    finetune_model = DativeCGR(
        model_name=new_model_name,
        seed=base_model.seed,
    )
    if not os.path.exists(finetune_model.model_path):
        finetune_model.load_from_checkpoint(base_model.model_path)
        weights = np.exp(
            -PAIRWISE_DISTANCE_FUNCTIONS["cosine"](
                np.concatenate(
                    [
                        fpgen.transform(dataset["donor_smiles"]),
                        fpgen.transform(dataset["acceptor_smiles"]),
                    ],
                    axis=1,
                ),
                np.concatenate(
                    [
                        fpgen.transform([target_donor]),
                        fpgen.transform([target_acceptor]),
                    ],
                    axis=1,
                ),
            )
        )
        finetune_model.fit(
            smis=train_smis.flatten(),
            y=(train_y * weights / weights.max()).repeat(
                len(rxn_smiles_columns), axis=0
            ),
            freeze=False,
            batch_size=batch_size,
            num_workers=num_workers,
            max_epochs=max_epochs,
        )
    return finetune_model

Loaded model from .checkpoints/augmented_base_model_reac_prod.pt.


/home/tmj/miniforge3/envs/feng/lib/python3.11/site-packages/chemprop/models/model.py:242: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(path, map_location)
/h

## P-1

In [3]:
target_acceptor = "O=C(C1=CC=CC=N1)/C=C/C2=CC=CC=C2"
target_donor = "CC[N+]([O-])=O"
target_product = "O=C(C1=CC=CC=N1)CC(C[N+]([O-])=O)C2=CC=CC=C2"
finetune_model = train_ood_model(
    base_model=base_model,
    new_model_tag="P-1",
    target_acceptor=target_acceptor,
    target_donor=target_donor,
    train_smis=train_smis,
    train_y=train_y,
    batch_size=batch_size,
    num_workers=16,
    max_epochs=200,
)
predictor = FengMichaelPredictor.load(finetune_model.model_path)
tuned_preds = pd.concat(
    [
        predictor.predict_michael(
            acceptor=target_acceptor,
            donor=target_donor,
            product=target_product,
            salt=[
                "[Sc+3].FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1])(F)F",
            ],
            ligand=["L3-RaPr2", "L3-PrPh"],
            solvent="CC[N+]([O-])=O",
            additive="CN(C)C1=CC=NC=C1",
            temperature=298,
            num_workers=10,
        ),
        predictor.predict_michael(
            acceptor=target_acceptor,
            donor=target_donor,
            product=target_product,
            salt=[
                "[La+3].FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1])(F)F",
            ],
            ligand="L3-PiPh",
            solvent="CC[N+]([O-])=O",
            additive="CN(C)C1=CC=NC=C1",
            temperature=298,
            num_workers=10,
        ),
    ],
    axis=0,
)
tuned_preds

Loaded model from .checkpoints/augmented_base_model_P-1_reac_prod.pt.


/home/tmj/miniforge3/envs/feng/lib/python3.11/site-packages/lightning/pytorch/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.5, which is newer than your current Lightning version: v2.4.0


Loaded model from /home/tmj/proj/NNdioxide-asymMichael/.checkpoints/augmented_base_model_P-1_reac_prod.pt.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

,salt,solvent,ligand,additive,acceptor,donor,product,ddG_estimation_mean,ddG_estimation_std,ee_estimation_mean,ee_estimation_std
0,[Sc+3].FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1...,CC[N+]([O-])=O,O=C([C@@H]1C[C@H]2[C@@H]([N@@+]1([O-1])CCC[N@+...,CN(C)C1=CC=NC=C1,O=C(C1=CC=CC=N1)/C=C/C2=CC=CC=C2,CC[N+]([O-])=O,O=C(C1=CC=CC=N1)CC(C[N+]([O-])=O)C2=CC=CC=C2,2.670014,0.010722,0.978217,0.000390
1,[Sc+3].FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1...,CC[N+]([O-])=O,O=C([C@@H]1CCCC[N@@+]1([O-1])CCC[N@+]1([O-1])C...,CN(C)C1=CC=NC=C1,O=C(C1=CC=CC=N1)/C=C/C2=CC=CC=C2,CC[N+]([O-])=O,O=C(C1=CC=CC=N1)CC(C[N+]([O-])=O)C2=CC=CC=C2,1.402026,0.016388,0.828608,0.004334
0,[La+3].FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1...,CC[N+]([O-])=O,O=C([C@@H]1CCCC[N@@+]1([O-1])CCC[N@+]1([O-1])C...,CN(C)C1=CC=NC=C1,O=C(C1=CC=CC=N1)/C=C/C2=CC=CC=C2,CC[N+]([O-])=O,O=C(C1=CC=CC=N1)CC(C[N+]([O-])=O)C2=CC=CC=C2,0.867390,0.005408,0.624525,0.002785


## P-2

In [4]:
target_acceptor = "O=C(C1=CC=CC=N1)/C=C/C2=CC=CC=C2"
target_donor = "O=C(NC1=C2C=CC=C1)C2CC3=CC=CC=C3"
target_product = "O=C(C1=CC=CC=N1)CC(C(C2=CC=CC=C2N3)(CC4=CC=CC=C4)C3=O)C5=CC=CC=C5"
finetune_model = train_ood_model(
    base_model=base_model,
    new_model_tag="P-2",
    target_acceptor=target_acceptor,
    target_donor=target_donor,
    train_smis=train_smis,
    train_y=train_y,
    batch_size=batch_size,
    num_workers=16,
    max_epochs=200,
)
predictor = FengMichaelPredictor.load(finetune_model.model_path)
tuned_preds = predictor.predict_michael(
    acceptor=target_acceptor,
    donor=target_donor,
    product=target_product,
    salt=[
        "[Yb+3].FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1])(F)F",
    ],
    ligand=["L3-PiPr2", "L3-PrAd"],
    solvent="ClCCl",
    additive=None,
    temperature=298,
    num_workers=10,
)
tuned_preds

/home/tmj/miniforge3/envs/feng/lib/python3.11/site-packages/chemprop/models/model.py:242: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(path, map_location)
/h

Loaded model from .checkpoints/augmented_base_model_P-2_reac_prod.pt.
Loaded model from /home/tmj/proj/NNdioxide-asymMichael/.checkpoints/augmented_base_model_P-2_reac_prod.pt.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

,salt,solvent,ligand,additive,acceptor,donor,product,ddG_estimation_mean,ddG_estimation_std,ee_estimation_mean,ee_estimation_std
0,[Yb+3].FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1...,ClCCl,O=C([C@@H]1CCCC[N@@+]1([O-1])CCC[N@+]1([O-1])C...,,O=C(C1=CC=CC=N1)/C=C/C2=CC=CC=C2,O=C(NC1=C2C=CC=C1)C2CC3=CC=CC=C3,O=C(C1=CC=CC=N1)CC(C(C2=CC=CC=C2N3)(CC4=CC=CC=...,1.458271,0.056141,0.842416,0.013727
1,[Yb+3].FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1...,ClCCl,O=C([C@@H]1CCC[N@@+]1([O-1])CCC[N@+]1([O-1])CC...,,O=C(C1=CC=CC=N1)/C=C/C2=CC=CC=C2,O=C(NC1=C2C=CC=C1)C2CC3=CC=CC=C3,O=C(C1=CC=CC=N1)CC(C(C2=CC=CC=C2N3)(CC4=CC=CC=...,0.866949,0.013617,0.624256,0.007018


## P-3

In [5]:
target_acceptor = "O=C(C1=CC=CC=N1)/C=C/C2=CC=CC=C2"
target_donor = "COC(S)=O"
target_product = "O=C(C1=CC=CC=N1)CC(SCC(OC)=O)C2=CC=CC=C2"
finetune_model = train_ood_model(
    base_model=base_model,
    new_model_tag="P-3",
    target_acceptor=target_acceptor,
    target_donor=target_donor,
    train_smis=train_smis,
    train_y=train_y,
    batch_size=batch_size,
    num_workers=16,
    max_epochs=200,
)
predictor = FengMichaelPredictor.load(finetune_model.model_path)
tuned_preds = predictor.predict_michael(
    acceptor=target_acceptor,
    donor=target_donor,
    product=target_product,
    salt=[
        "[Sc+3].FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1])(F)F",
    ],
    ligand=["L3-PrPh"],
    solvent="ClCCCl",
    additive=None,
    temperature=298,
    num_workers=10,
)
tuned_preds

/home/tmj/miniforge3/envs/feng/lib/python3.11/site-packages/chemprop/models/model.py:242: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(path, map_location)
/h

Loaded model from .checkpoints/augmented_base_model_P-3_reac_prod.pt.
Loaded model from /home/tmj/proj/NNdioxide-asymMichael/.checkpoints/augmented_base_model_P-3_reac_prod.pt.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

,salt,solvent,ligand,additive,acceptor,donor,product,ddG_estimation_mean,ddG_estimation_std,ee_estimation_mean,ee_estimation_std
0,[Sc+3].FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1...,ClCCCl,O=C([C@@H]1CCCC[N@@+]1([O-1])CCC[N@+]1([O-1])C...,,O=C(C1=CC=CC=N1)/C=C/C2=CC=CC=C2,COC(S)=O,O=C(C1=CC=CC=N1)CC(SCC(OC)=O)C2=CC=CC=C2,0.526461,0.017793,0.417315,0.012419


## P-4

In [6]:
target_acceptor = "Cc1cc(C)n(C(/C=C/c2ccccc2)=O)n1"
target_donor = "COC(CS)=O"
target_product = "COC(CSC(CC(n1c(C)cc(C)n1)=O)c2ccccc2)=O"
predictor = FengMichaelPredictor.load(
    ".checkpoints/augmented_weighted_tuned_model_reac_prod.pt"
)
tuned_preds = predictor.predict_michael(
    acceptor=target_acceptor,
    donor=target_donor,
    product=target_product,
    salt=[
        "[Yb+3].FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1])(F)F",
    ],
    ligand=["L3-PrPh", "L3-PiPh"],
    solvent="ClC(Cl)Cl",
    additive="[K+].[O-1]C(=O)[O-1].[K+]",
    temperature=298,
    num_workers=10,
)
tuned_preds

/home/tmj/miniforge3/envs/feng/lib/python3.11/site-packages/chemprop/models/model.py:242: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(path, map_location)
/h

Loaded model from /home/tmj/proj/NNdioxide-asymMichael/.checkpoints/augmented_weighted_tuned_model_reac_prod.pt.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

,salt,solvent,ligand,additive,acceptor,donor,product,ddG_estimation_mean,ddG_estimation_std,ee_estimation_mean,ee_estimation_std
0,[Yb+3].FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1...,ClC(Cl)Cl,O=C([C@@H]1CCCC[N@@+]1([O-1])CCC[N@+]1([O-1])C...,[K+].[O-1]C(=O)[O-1].[K+],Cc1cc(C)n(C(/C=C/c2ccccc2)=O)n1,COC(CS)=O,COC(CSC(CC(n1c(C)cc(C)n1)=O)c2ccccc2)=O,0.082571,0.00443,0.069607,0.003723
1,[Yb+3].FC(S(=O)(=O)[O-1])(F)F.FC(S(=O)(=O)[O-1...,ClC(Cl)Cl,O=C([C@@H]1CCCC[N@@+]1([O-1])CCC[N@+]1([O-1])C...,[K+].[O-1]C(=O)[O-1].[K+],Cc1cc(C)n(C(/C=C/c2ccccc2)=O)n1,COC(CS)=O,COC(CSC(CC(n1c(C)cc(C)n1)=O)c2ccccc2)=O,0.082571,0.00443,0.069607,0.003723
